# Lean-16d : Game of Life sur kernel Lean natif

**Navigation** : [<< Lean-16b GOL (Python+Lean)](Lean-16b-Conway-Game-of-Life-Lean.ipynb) | [Index](README.md) | [Lean-16c Compagnon Golly >>](Lean-16c-Conway-Game-of-Life-Golly.ipynb) | [Lean-13 Kochen-Specker >>](Lean-13-Kochen-Specker.ipynb)

**Kernel** : **Lean 4 (WSL) natif** — chaque cellule de code est du Lean 4 exécuté directement, sans intermédiaire Python.

> Ce notebook complète [Lean-16b](Lean-16b-Conway-Game-of-Life-Lean.ipynb) qui présente le *récit* et les *illustrations* du Game of Life via un kernel Python. Ici, à l'inverse, **l'étudiant vit Lean comme un REPL interactif** : on définit la grille, la règle B3/S23, le moteur d'évolution, puis on démontre et certifie de petits faits — le tout dans des cellules Lean évaluées en place.


## 1. Pourquoi un kernel Lean natif ?

Le sous-titre de la série Conway est *« source de vérité formelle »*. Pourtant, jusqu'ici, le formalisme Lean était **encapsulé derrière des appels Python** (`run_lake`, `run_lean_snippet`) dans [Lean-16b](Lean-16b-Conway-Game-of-Life-Lean.ipynb). L'étudiant n'interagissait jamais avec Lean directement.

Ce notebook corrige cette incohérence : on utilise le **kernel `lean4-wsl`**, qui exécute chaque cellule comme une commande Lean 4 dans un REPL. C'est la même expérience que les notebooks [GameTheory-2b](../../GameTheory/GameTheory-2b-Lean-Definitions.ipynb), `4b`, `8b`, `15b`.

**Vérifions que le kernel répond** (deux diagnostics standard) :

In [1]:
#eval 2 + 2
#check Nat


#eval 2 + 2
─────▶  4
#check Nat
──────▶  Nat : Type

--% env 0

Raw input:
{"cmd": "#eval 2 + 2\n#check Nat\n"}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 5},
   "data": "4"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data": "Nat : Type"}],
 "env": 0}


## 2. Représentation de la grille

Le Game of Life se joue sur le plan entier $\mathbb{Z}^2$. Une cellule est une paire d'entiers `(Int × Int)`. Une **grille** est la liste des cellules vivantes (l'ordre importe peu ; les cellules absentes sont mortes).

On choisit `List (Int × Int)` plutôt qu'un `Finset` car l'égalité de listes se réduit à une comparaison structurelle — les tactiques de décision (`decide`, `native_decide`) et le code natif la manipulent efficacement, sans le goulet `Quot.lift` des `Finset`.


In [2]:
-- Une grille = liste des cellules vivantes
abbrev Grid := List (Int × Int)

-- Les 8 voisins de Moore (déplacements du roi aux échecs) d'une cellule
def mooreNeighbors (p : Int × Int) : List (Int × Int) :=
  [(p.1 - 1, p.2 - 1), (p.1 - 1, p.2), (p.1 - 1, p.2 + 1),
   (p.1, p.2 - 1),                 (p.1, p.2 + 1),
   (p.1 + 1, p.2 - 1), (p.1 + 1, p.2), (p.1 + 1, p.2 + 1)]

#eval mooreNeighbors (0, 0)


-- Une grille = liste des cellules vivantes
abbrev Grid := List (Int × Int)

-- Les 8 voisins de Moore (déplacements du roi aux échecs) d'une cellule
def mooreNeighbors (p : Int × Int) : List (Int × Int) :=
  [(p.1 - 1, p.2 - 1), (p.1 - 1, p.2), (p.1 - 1, p.2 + 1),
   (p.1, p.2 - 1),                 (p.1, p.2 + 1),
   (p.1 + 1, p.2 - 1), (p.1 + 1, p.2), (p.1 + 1, p.2 + 1)]

#eval mooreNeighbors (0, 0)
─────▶  [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]

--% env 1

Raw input:
{"cmd": "-- Une grille = liste des cellules vivantes\nabbrev Grid := List (Int \u00d7 Int)\n\n-- Les 8 voisins de Moore (d\u00e9placements du roi aux \u00e9checs) d'une cellule\ndef mooreNeighbors (p : Int \u00d7 Int) : List (Int \u00d7 Int) :=\n  [(p.1 - 1, p.2 - 1), (p.1 - 1, p.2), (p.1 - 1, p.2 + 1),\n   (p.1, p.2 - 1),                 (p.1, p.2 + 1),\n   (p.1 + 1, p.2 - 1), (p.1 + 1, p.2), (p.1 + 1, p.2 + 1)]\n\n#eval mooreNeighbors (0, 0)\n", "env": 0}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 10, "column": 0},
   "endPos": {"line": 10, "column": 5},
   "data":
   "[(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]"}],
 "env": 1}


## 3. La règle B3/S23

Pour chaque cellule, on compte ses voisins vivants :
- **Birth (B3)** : une cellule **morte** avec exactement 3 voisins vivants naît ;
- **Survival (S23)** : une cellule **vivante** avec 2 ou 3 voisins vivants survit ;
- sinon la cellule est morte à la génération suivante.


In [3]:
-- La cellule `p` est-elle vivante dans `g` ?
def isAlive (g : Grid) (p : Int × Int) : Bool := g.contains p

-- Nombre de voisins vivants de `p` dans `g`
def liveNeighborCount (g : Grid) (p : Int × Int) : Nat :=
  (mooreNeighbors p).countP (isAlive g)

-- La règle B3/S23 : `p` doit-elle être vivante à la génération suivante ?
def aliveNext (g : Grid) (p : Int × Int) : Bool :=
  let n := liveNeighborCount g p
  if isAlive g p then n == 2 || n == 3   -- S23
  else n == 3                            -- B3

-- Un oscillateur simple : le clignoteur (blinker), 3 cellules en ligne
def blinker : Grid := [(0,0), (0,1), (0,2)]

-- La cellule centrale (0,1) a ses 2 voisins vivants => elle survit
#eval liveNeighborCount blinker (0, 1)
#eval aliveNext blinker (0, 1)


-- La cellule `p` est-elle vivante dans `g` ?
def isAlive (g : Grid) (p : Int × Int) : Bool := g.contains p

-- Nombre de voisins vivants de `p` dans `g`
def liveNeighborCount (g : Grid) (p : Int × Int) : Nat :=
  (mooreNeighbors p).countP (isAlive g)

-- La règle B3/S23 : `p` doit-elle être vivante à la génération suivante ?
def aliveNext (g : Grid) (p : Int × Int) : Bool :=
  let n := liveNeighborCount g p
  if isAlive g p then n == 2 || n == 3   -- S23
  else n == 3                            -- B3

-- Un oscillateur simple : le clignoteur (blinker), 3 cellules en ligne
def blinker : Grid := [(0,0), (0,1), (0,2)]

-- La cellule centrale (0,1) a ses 2 voisins vivants => elle survit
#eval liveNeighborCount blinker (0, 1)
─────▶  2
#eval aliveNext blinker (0, 1)
─────▶  true

--% env 2

Raw input:
{"cmd": "-- La cellule `p` est-elle vivante dans `g` ?\ndef isAlive (g : Grid) (p : Int \u00d7 Int) : Bool := g.contains p\n\n-- Nombre de voisins vivants de `p` dans `g`\ndef liveNeighborCount (g : Grid) (p : Int \u00d7 Int) : Nat :=\n  (mooreNeighbors p).countP (isAlive g)\n\n-- La r\u00e8gle B3/S23 : `p` doit-elle \u00eatre vivante \u00e0 la g\u00e9n\u00e9ration suivante ?\ndef aliveNext (g : Grid) (p : Int \u00d7 Int) : Bool :=\n  let n := liveNeighborCount g p\n  if isAlive g p then n == 2 || n == 3   -- S23\n  else n == 3                            -- B3\n\n-- Un oscillateur simple : le clignoteur (blinker), 3 cellules en ligne\ndef blinker : Grid := [(0,0), (0,1), (0,2)]\n\n-- La cellule centrale (0,1) a ses 2 voisins vivants => elle survit\n#eval liveNeighborCount blinker (0, 1)\n#eval aliveNext blinker (0, 1)\n", "env": 1}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 18, "column": 0},
   "endPos": {"line": 18, "column": 5},
   "data": "2"},
  {"severity": "info",
   "pos": {"line": 19, "column": 0},
   "endPos": {"line": 19, "column": 5},
   "data": "true"}],
 "env": 2}


## 4. Le moteur d'évolution

Une étape (`step`) : on rassemble les cellules candidates (vivantes + leurs voisins), on garde celles qui satisfont `aliveNext`, et on déduplique. Puis `evolve` itère `step`.


In [4]:
-- Cellules candidates = vivantes + tous leurs voisins (mortes incluses)
def candidates (g : Grid) : List (Int × Int) :=
  g ++ g.flatMap mooreNeighbors

-- Une génération de Game of Life
def step (g : Grid) : Grid :=
  ((candidates g).filter (aliveNext g)).eraseDups

-- Itération : `evolve g n` applique `n` fois la règle
def evolve (g : Grid) : Nat → Grid
  | 0 => g
  | n + 1 => step (evolve g n)

-- Le clignoteur bascule d'horizontal à vertical
#eval step blinker
#eval step (step blinker)   -- retour à la ligne horizontale : période 2


-- Cellules candidates = vivantes + tous leurs voisins (mortes incluses)
def candidates (g : Grid) : List (Int × Int) :=
  g ++ g.flatMap mooreNeighbors

-- Une génération de Game of Life
def step (g : Grid) : Grid :=
  ((candidates g).filter (aliveNext g)).eraseDups

-- Itération : `evolve g n` applique `n` fois la règle
def evolve (g : Grid) : Nat → Grid
  | 0 => g
  | n + 1 => step (evolve g n)

-- Le clignoteur bascule d'horizontal à vertical
#eval step blinker
─────▶  [(0, 1), (-1, 1), (1, 1)]
#eval step (step blinker)   -- retour à la ligne horizontale : période 2
─────▶  [(0, 1), (0, 0), (0, 2)]

--% env 3

Raw input:
{"cmd": "-- Cellules candidates = vivantes + tous leurs voisins (mortes incluses)\ndef candidates (g : Grid) : List (Int \u00d7 Int) :=\n  g ++ g.flatMap mooreNeighbors\n\n-- Une g\u00e9n\u00e9ration de Game of Life\ndef step (g : Grid) : Grid :=\n  ((candidates g).filter (aliveNext g)).eraseDups\n\n-- It\u00e9ration : `evolve g n` applique `n` fois la r\u00e8gle\ndef evolve (g : Grid) : Nat \u2192 Grid\n  | 0 => g\n  | n + 1 => step (evolve g n)\n\n-- Le clignoteur bascule d'horizontal \u00e0 vertical\n#eval step blinker\n#eval step (step blinker)   -- retour \u00e0 la ligne horizontale : p\u00e9riode 2\n", "env": 2}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 15, "column": 0},
   "endPos": {"line": 15, "column": 5},
   "data": "[(0, 1), (-1, 1), (1, 1)]"},
  {"severity": "info",
   "pos": {"line": 16, "column": 0},
   "endPos": {"line": 16, "column": 5},
   "data": "[(0, 1), (0, 0), (0, 2)]"}],
 "env": 3}


## 5. Configurations stables (still-lifes)

Un **still-life** est une figure qui ne bouge pas : `step g = g`. Le **bloc** (carré 2×2) et la **ruche** (beehive) sont les exemples canoniques.


In [5]:
-- Bloc : carré 2x2. Chaque cellule a 3 voisins vivants => survit (S23).
def block : Grid := [(0,0), (0,1), (1,0), (1,1)]

-- Ruche : still-life à 6 cellules
def beehive : Grid := [(0,1), (0,2), (1,0), (1,3), (2,1), (2,2)]

#eval step block       -- inchangé (au tri près)
#eval liveNeighborCount block (0, 0)


-- Bloc : carré 2x2. Chaque cellule a 3 voisins vivants => survit (S23).
def block : Grid := [(0,0), (0,1), (1,0), (1,1)]

-- Ruche : still-life à 6 cellules
def beehive : Grid := [(0,1), (0,2), (1,0), (1,3), (2,1), (2,2)]

#eval step block       -- inchangé (au tri près)
─────▶  [(0, 0), (0, 1), (1, 0), (1, 1)]
#eval liveNeighborCount block (0, 0)
─────▶  3

--% env 4

Raw input:
{"cmd": "-- Bloc : carr\u00e9 2x2. Chaque cellule a 3 voisins vivants => survit (S23).\ndef block : Grid := [(0,0), (0,1), (1,0), (1,1)]\n\n-- Ruche : still-life \u00e0 6 cellules\ndef beehive : Grid := [(0,1), (0,2), (1,0), (1,3), (2,1), (2,2)]\n\n#eval step block       -- inchang\u00e9 (au tri pr\u00e8s)\n#eval liveNeighborCount block (0, 0)\n", "env": 3}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 5},
   "data": "[(0, 0), (0, 1), (1, 0), (1, 1)]"},
  {"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 5},
   "data": "3"}],
 "env": 4}


## 6. Oscillateurs

Un **oscillateur** de période $p$ vérifie `evolve g p = g`. Le clignoteur est de période 2 ; le **crapaud** (toad) et le **phare** (beacon) aussi.


In [6]:
-- Crapeau (toad) : période 2
def toad : Grid := [(0,1), (0,2), (0,3), (1,0), (1,1), (1,2)]

-- Phare (beacon) : période 2
def beacon : Grid := [(0,0), (0,1), (1,0), (1,1), (2,2), (2,3), (3,2), (3,3)]

-- L'ensemble des cellules est préservé après 2 pas (set-égalité)
def sameSet (a b : Grid) : Bool := a.all (fun p => b.contains p) && b.all (fun p => a.contains p)

#eval sameSet (evolve blinker 2) blinker
#eval sameSet (evolve toad 2) toad
#eval sameSet (evolve beacon 2) beacon


-- Crapeau (toad) : période 2
def toad : Grid := [(0,1), (0,2), (0,3), (1,0), (1,1), (1,2)]

-- Phare (beacon) : période 2
def beacon : Grid := [(0,0), (0,1), (1,0), (1,1), (2,2), (2,3), (3,2), (3,3)]

-- L'ensemble des cellules est préservé après 2 pas (set-égalité)
def sameSet (a b : Grid) : Bool := a.all (fun p => b.contains p) && b.all (fun p => a.contains p)

#eval sameSet (evolve blinker 2) blinker
─────▶  true
#eval sameSet (evolve toad 2) toad
─────▶  true
#eval sameSet (evolve beacon 2) beacon
─────▶  true

--% env 5

Raw input:
{"cmd": "-- Crapeau (toad) : p\u00e9riode 2\ndef toad : Grid := [(0,1), (0,2), (0,3), (1,0), (1,1), (1,2)]\n\n-- Phare (beacon) : p\u00e9riode 2\ndef beacon : Grid := [(0,0), (0,1), (1,0), (1,1), (2,2), (2,3), (3,2), (3,3)]\n\n-- L'ensemble des cellules est pr\u00e9serv\u00e9 apr\u00e8s 2 pas (set-\u00e9galit\u00e9)\ndef sameSet (a b : Grid) : Bool := a.all (fun p => b.contains p) && b.all (fun p => a.contains p)\n\n#eval sameSet (evolve blinker 2) blinker\n#eval sameSet (evolve toad 2) toad\n#eval sameSet (evolve beacon 2) beacon\n", "env": 4}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 10, "column": 0},
   "endPos": {"line": 10, "column": 5},
   "data": "true"},
  {"severity": "info",
   "pos": {"line": 11, "column": 0},
   "endPos": {"line": 11, "column": 5},
   "data": "true"},
  {"severity": "info",
   "pos": {"line": 12, "column": 0},
   "endPos": {"line": 12, "column": 5},
   "data": "true"}],
 "env": 5}


## 7. Vaisseau (glider)

Un **vaisseau** (spaceship) se déplace. Le **planeur** (glider), découvert par Richard Guy en 1970, parcourt la diagonale : après 4 générations il est translaté de `(1, 1)`.


In [7]:
-- Planeur : 5 cellules, se déplace en diagonale
def glider : Grid := [(0,1), (1,2), (2,0), (2,1), (2,2)]

-- Translation d'une grille par (dr, dc)
def translate (dr dc : Int) (g : Grid) : Grid :=
  g.map (fun (r, c) => (r + dr, c + dc))

#eval sameSet (evolve glider 4) (translate 1 1 glider)   -- le planeur avance


-- Planeur : 5 cellules, se déplace en diagonale
def glider : Grid := [(0,1), (1,2), (2,0), (2,1), (2,2)]

-- Translation d'une grille par (dr, dc)
def translate (dr dc : Int) (g : Grid) : Grid :=
  g.map (fun (r, c) => (r + dr, c + dc))

#eval sameSet (evolve glider 4) (translate 1 1 glider)   -- le planeur avance
─────▶  true

--% env 6

Raw input:
{"cmd": "-- Planeur : 5 cellules, se d\u00e9place en diagonale\ndef glider : Grid := [(0,1), (1,2), (2,0), (2,1), (2,2)]\n\n-- Translation d'une grille par (dr, dc)\ndef translate (dr dc : Int) (g : Grid) : Grid :=\n  g.map (fun (r, c) => (r + dr, c + dc))\n\n#eval sameSet (evolve glider 4) (translate 1 1 glider)   -- le planeur avance\n", "env": 5}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 8, "column": 0},
   "endPos": {"line": 8, "column": 5},
   "data": "true"}],
 "env": 6}


## 8. Petits faits certifiés

L'avantage d'un noyau formel : on **prouve**, pas seulement on observe. `decide` demande à Lean si un énoncé décidable clos est vrai ; `native_decide` fait de même via le code natif (plus rapide sur des domaines plus grands).


In [8]:
-- Le centre du clignoteur a bien 2 voisins vivants
theorem blinker_center_has_two_neighbors :
    liveNeighborCount blinker (0, 1) = 2 := by decide

-- Une cellule vivante du bloc a 3 voisins => elle survit (S23)
theorem block_cell_survives :
    aliveNext block (0, 0) = true := by decide

-- Une cellule isolée n'a aucun voisin => elle meurt (0 ≠ 2, 0 ≠ 3)
theorem lonely_cell_dies :
    aliveNext [(5, 5)] (5, 5) = false := by decide

-- Le clignoteur est un oscillateur de période 2 (set-égalité, native_decide)
theorem blinker_period_two :
    sameSet (evolve blinker 2) blinker = true := by native_decide


-- Le centre du clignoteur a bien 2 voisins vivants
theorem blinker_center_has_two_neighbors :
    liveNeighborCount blinker (0, 1) = 2 := by decide

-- Une cellule vivante du bloc a 3 voisins => elle survit (S23)
theorem block_cell_survives :
    aliveNext block (0, 0) = true := by decide

-- Une cellule isolée n'a aucun voisin => elle meurt (0 ≠ 2, 0 ≠ 3)
theorem lonely_cell_dies :
    aliveNext [(5, 5)] (5, 5) = false := by decide

-- Le clignoteur est un oscillateur de période 2 (set-égalité, native_decide)
theorem blinker_period_two :
    sameSet (evolve blinker 2) blinker = true := by native_decide

--% env 7

Raw input:
{"cmd": "-- Le centre du clignoteur a bien 2 voisins vivants\ntheorem blinker_center_has_two_neighbors :\n    liveNeighborCount blinker (0, 1) = 2 := by decide\n\n-- Une cellule vivante du bloc a 3 voisins => elle survit (S23)\ntheorem block_cell_survives :\n    aliveNext block (0, 0) = true := by decide\n\n-- Une cellule isol\u00e9e n'a aucun voisin => elle meurt (0 \u2260 2, 0 \u2260 3)\ntheorem lonely_cell_dies :\n    aliveNext [(5, 5)] (5, 5) = false := by decide\n\n-- Le clignoteur est un oscillateur de p\u00e9riode 2 (set-\u00e9galit\u00e9, native_decide)\ntheorem blinker_period_two :\n    sameSet (evolve blinker 2) blinker = true := by native_decide\n", "env": 6}
Raw output:
{"env": 7}


## 9. La frontière du prouvé : transparence

Les théorèmes ci-dessus ne dépendent **d'aucun axiome caché** — en particulier pas de `sorry`. Vérifions-le :


In [9]:
#print axioms blinker_center_has_two_neighbors
#print axioms blinker_period_two


#print axioms blinker_center_has_two_neighbors
──────▶  'blinker_center_has_two_neighbors' does not depend on any axioms
#print axioms blinker_period_two
──────▶  'blinker_period_two' depends on axioms: [blinker_period_two._native.native_decide.ax_1]

--% env 8

Raw input:
{"cmd": "#print axioms blinker_center_has_two_neighbors\n#print axioms blinker_period_two\n", "env": 7}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 1, "column": 0},
   "endPos": {"line": 1, "column": 6},
   "data": "'blinker_center_has_two_neighbors' does not depend on any axioms"},
  {"severity": "info",
   "pos": {"line": 2, "column": 0},
   "endPos": {"line": 2, "column": 6},
   "data":
   "'blinker_period_two' depends on axioms: [blinker_period_two._native.native_decide.ax_1]"}],
 "env": 8}


> Les théorèmes ci-dessus ne sont pas une formalisation complète du Game of Life : ils certifient des **faits locaux** (comptages, survie) sur des **petites grilles**. La formalisation complète — avec l'optimisation *Hashlife* de Gosper (1984), le découpage en macro-cellules, et la preuve que `hashlifeJump` simule correctement $2^k$ générations — vit dans le projet `conway_lean/` du dépôt (modules `Conway.Life.Hashlife`, `Conway.Life.HashlifeCorrectness`).
>
> **Frontière assumée** : `HashlifeCorrectness.lean` contient encore **2 lemmes admis** (`sorry`), sur la partie la plus subtile du résultat (composition inductive des macro-cellules). C'est la limite actuelle, ouvertement documentée, du travail de formalisation. Ce notebook se concentre sur ce qui est **prouvé de bout en bout**.



## 10. Exercices

Trois exercices pour manipuler directement le moteur. Chaque cellule ci-dessous **compile et s'exécute** dans l'état (corps de substitution) ; à vous de remplacer la partie signalée `-- TODO`.


### Exercice 1 — Compter sans `countP`

Réécrivez `liveNeighborCount` à l'aide d'un `foldl` sur la liste des voisins, **sans** utiliser `countP`. *Indice : accumuler un `Nat` qui s'incrémente de 1 pour chaque voisin vivant.*


In [10]:
-- Exercice 1 : compter les voisins vivants de `p` dans `g` via foldl
-- TODO étudiant : remplacer le corps ci-dessous
def liveNeighborCountFold (g : Grid) (p : Int × Int) : Nat :=
  0

-- Doit donner 2 pour le centre du clignoteur
#eval liveNeighborCountFold blinker (0, 1)


-- Exercice 1 : compter les voisins vivants de `p` dans `g` via foldl
-- TODO étudiant : remplacer le corps ci-dessous
def liveNeighborCountFold (g : Grid) (p : Int × Int) : Nat :=
                           ─▶ 🟨 unused variable `g`

Note: This linter can be disabled with `set_option linter.unusedVariables false`
                                      ─▶ 🟨 unused variable `p`

Note: This linter can be disabled with `set_option linter.unusedVariables false`
  0

-- Doit donner 2 pour le centre du clignoteur
#eval liveNeighborCountFold blinker (0, 1)
─────▶  0

--% env 9

Raw input:
{"cmd": "-- Exercice 1 : compter les voisins vivants de `p` dans `g` via foldl\n-- TODO \u00e9tudiant : remplacer le corps ci-dessous\ndef liveNeighborCountFold (g : Grid) (p : Int \u00d7 Int) : Nat :=\n  0\n\n-- Doit donner 2 pour le centre du clignoteur\n#eval liveNeighborCountFold blinker (0, 1)\n", "env": 8}
Raw output:
{"messages":
 [{"severity": "warning",
   "pos": {"line": 3, "column": 27},
   "endPos": {"line": 3, "column": 28},
   "data":
   "unused variable `g`\n\nNote: This linter can be disabled with `set_option linter.unusedVariables false`"},
  {"severity": "warning",
   "pos": {"line": 3, "column": 38},
   "endPos": {"line": 3, "column": 39},
   "data":
   "unused variable `p`\n\nNote: This linter can be disabled with `set_option linter.unusedVariables false`"},
  {"severity": "info",
   "pos": {"line": 7, "column": 0},
   "endPos": {"line": 7, "column": 5},
   "data": "0"}],
 "env": 9}


### Exercice 2 — Prouver un fait local

Démontrez que la cellule morte `(1, 1)` naît bien à la génération suivante du clignoteur (elle a 3 voisins vivants). Remplacez `sorry` par une tactique. *Indice : `decide` ou `native_decide`.*


In [11]:
-- Exercice 2 : (1,1) est morte dans blinker, a 3 voisins vivants => naît
-- TODO étudiant : remplacer sorry
theorem blinker_birth_at_11 :
    aliveNext blinker (1, 1) = true := by sorry


-- Exercice 2 : (1,1) est morte dans blinker, a 3 voisins vivants => naît
-- TODO étudiant : remplacer sorry
theorem blinker_birth_at_11 :
        ───────────────────▶ 🟨 declaration uses `sorry`
    aliveNext blinker (1, 1) = true := by sorry

--% env 10
--% prove 0

Raw input:
{"cmd": "-- Exercice 2 : (1,1) est morte dans blinker, a 3 voisins vivants => na\u00eet\n-- TODO \u00e9tudiant : remplacer sorry\ntheorem blinker_birth_at_11 :\n    aliveNext blinker (1, 1) = true := by sorry\n", "env": 9}
Raw output:
{"sorries":
 [{"proofState": 0,
   "pos": {"line": 4, "column": 42},
   "goal": "⊢ aliveNext blinker (1, 1) = true",
   "endPos": {"line": 4, "column": 47}}],
 "messages":
 [{"severity": "warning",
   "pos": {"line": 3, "column": 8},
   "endPos": {"line": 3, "column": 27},
   "data": "declaration uses `sorry`"}],
 "env": 10}


### Exercice 3 — Un nouvel oscillateur

Définissez le **pulsar** miniature ci-dessous et vérifiez qu'il est de période 2 (set-égalité avec `sameSet`). *Indice : inspirez-vous des définitions de `blinker`/`toad`.*


In [12]:
-- Exercice 3 : définir un oscillateur et vérifier sa période
-- TODO étudiant : remplacer le corps ci-dessous par une vraie figure
def myOscillator : Grid := [(0, 0)]

#eval sameSet (evolve myOscillator 2) myOscillator


-- Exercice 3 : définir un oscillateur et vérifier sa période
-- TODO étudiant : remplacer le corps ci-dessous par une vraie figure
def myOscillator : Grid := [(0, 0)]

#eval sameSet (evolve myOscillator 2) myOscillator
─────▶  false

--% env 11

Raw input:
{"cmd": "-- Exercice 3 : d\u00e9finir un oscillateur et v\u00e9rifier sa p\u00e9riode\n-- TODO \u00e9tudiant : remplacer le corps ci-dessous par une vraie figure\ndef myOscillator : Grid := [(0, 0)]\n\n#eval sameSet (evolve myOscillator 2) myOscillator\n", "env": 10}
Raw output:
{"messages":
 [{"severity": "info",
   "pos": {"line": 5, "column": 0},
   "endPos": {"line": 5, "column": 5},
   "data": "false"}],
 "env": 11}


## Conclusion

Sur le kernel Lean natif, l'étudiant a : (1) **défini** le Game of Life (grille, voisinage, règle B3/S23, moteur `step`/`evolve`) en pur Lean 4 ; (2) **observé** les motifs canoniques (bloc, clignoteur, planeur) via `#eval` ; (3) **certifié** des faits locaux par `decide`/`native_decide` sans axiome `sorry`.

Pour la formalisation complète (Hashlife, macro-cellules, universalité), voir le projet [`conway_lean/`](https://github.com/jsboige/CoursIA) et le notebook compagnon [Lean-16b](Lean-16b-Conway-Game-of-Life-Lean.ipynb). La frontière du prouvé y est assumée : 2 lemmes de `HashlifeCorrectness` restent admis, et font l'objet d'un travail de preuve en cours.

**Suite logique** : [Lean-13 Kochen-Specker](Lean-13-Kochen-Specker.ipynb) — un autre résultat de Conway (le théorème du libre arbitre), cette fois-ci **entièrement prouvé** (0 `sorry`).
